# AWS S3 and Athena Integration

Working with AWS S3 for storage and Athena for querying.

In [ ]:
import boto3
import pandas as pd
import time
from pathlib import Path

## S3 Operations

In [ ]:
# Initialize S3 client
s3_client = boto3.client('s3', region_name='us-east-1')

# Configuration
bucket_name = 'my-data-bucket'  # Replace with your bucket
region = 'us-east-1'

print(f"S3 client configured for region: {region}")

In [ ]:
# List S3 buckets
try:
    response = s3_client.list_buckets()
    buckets = [b['Name'] for b in response['Buckets']]
    print("Available buckets:")
    for bucket in buckets:
        print(f"  - {bucket}")
except Exception as e:
    print(f"Error listing buckets: {e}")
    print("Make sure your AWS credentials are configured.")

## Upload Data to S3

In [ ]:
# Create sample data
df = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'amount': [100, 250, 75, 125, 200]
})

# Save locally first
local_path = '../data/processed/sample_data.parquet'
df.to_parquet(local_path, index=False)
print(f"Data saved to {local_path}")

In [ ]:
# Upload to S3
try:
    s3_key = 'data/processed/sample_data.parquet'
    s3_client.upload_file(local_path, bucket_name, s3_key)
    print(f"Uploaded to s3://{bucket_name}/{s3_key}")
except Exception as e:
    print(f"Error uploading to S3: {e}")
    print(f"Make sure the bucket '{bucket_name}' exists and you have write permissions.")

## Athena Queries

In [ ]:
# Initialize Athena client
athena_client = boto3.client('athena', region_name=region)

# Configuration
database = 'default'
results_location = f's3://{bucket_name}/athena-results/'

print(f"Athena configured:")
print(f"  Database: {database}")
print(f"  Results location: {results_location}")

In [ ]:
def run_athena_query(query, database, results_location):
    """Execute an Athena query and return results."""
    try:
        # Start query execution
        response = athena_client.start_query_execution(
            QueryString=query,
            QueryExecutionContext={'Database': database},
            ResultConfiguration={'OutputLocation': results_location}
        )
        
        query_id = response['QueryExecutionId']
        print(f"Query submitted. ID: {query_id}")
        
        # Wait for query to complete
        while True:
            query_status = athena_client.get_query_execution(QueryExecutionId=query_id)
            status = query_status['QueryExecution']['Status']['State']
            
            if status == 'SUCCEEDED':
                print(f"Query succeeded!")
                break
            elif status in ['FAILED', 'CANCELLED']:
                print(f"Query {status}")
                return None
            
            time.sleep(1)
        
        # Get results
        results = athena_client.get_query_results(QueryExecutionId=query_id)
        return results
    
    except Exception as e:
        print(f"Error executing query: {e}")
        return None

print("Query execution function defined.")

In [ ]:
# Example: Simple query to test connectivity
test_query = "SHOW DATABASES;"

try:
    results = run_athena_query(test_query, database, results_location)
    if results:
        print("\nQueryable tables/databases:")
        rows = results['ResultSet']['Rows']
        for row in rows[1:]:  # Skip header
            print(f"  {row['Data'][0]['VarCharValue']}")
except Exception as e:
    print(f"Error: {e}")

## Creating External Tables

In [ ]:
# Create external table pointing to S3 data
create_table_query = f"""
    CREATE EXTERNAL TABLE IF NOT EXISTS sample_data (
        id INT,
        name STRING,
        amount DOUBLE
    )
    STORED AS PARQUET
    LOCATION 's3://{bucket_name}/data/processed/'
    """

print("Create table query:")
print(create_table_query)

In [ ]:
# Execute create table
try:
    results = run_athena_query(create_table_query, database, results_location)
    if results:
        print("External table created successfully!")
except Exception as e:
    print(f"Error creating table: {e}")

## Querying with Athena

In [ ]:
# Query the external table
select_query = "SELECT * FROM sample_data LIMIT 10;"

try:
    results = run_athena_query(select_query, database, results_location)
    
    if results:
        # Parse results
        rows = results['ResultSet']['Rows']
        
        # Get headers
        headers = [col['Name'] for col in results['ResultSet']['ResultSetMetadata']['ColumnInfo']]
        print(f"Headers: {headers}")
        
        # Get data rows
        print("\nData:")
        for row in rows[1:]:  # Skip header row
            values = [d.get('VarCharValue', '') for d in row['Data']]
            print(values)
except Exception as e:
    print(f"Error querying: {e}")

## Aggregation Queries

In [ ]:
# Aggregation query
agg_query = """
    SELECT 
        COUNT(*) as total_records,
        SUM(amount) as total_amount,
        AVG(amount) as avg_amount
    FROM sample_data
    """

try:
    results = run_athena_query(agg_query, database, results_location)
    
    if results:
        rows = results['ResultSet']['Rows']
        print("Aggregation Results:")
        print(rows[0]['Data'][0]['VarCharValue'])
        # Parse and display results
        if len(rows) > 1:
            data = rows[1]['Data']
            print(f"  Total Records: {data[0].get('VarCharValue', 'N/A')}")
            print(f"  Total Amount: {data[1].get('VarCharValue', 'N/A')}")
            print(f"  Avg Amount: {data[2].get('VarCharValue', 'N/A')}")
except Exception as e:
    print(f"Error: {e}")

## Best Practices

In [ ]:
print("""
    AWS S3 and Athena Best Practices:
    
    1. Data Organization:
       - Use partitioning (year/month/day)
       - Organize by data source
       - Separate raw and processed data
    
    2. File Format:
       - Use Parquet or ORC
       - Compress with snappy or gzip
       - Aim for 128MB+ files
    
    3. Query Optimization:
       - Use partition pruning
       - Select only needed columns
       - Use WHERE clauses to limit data
    
    4. Cost Management:
       - First 1GB per month is free
       - $6.25 per 1TB scanned
       - Query efficient DDL/DML
    
    5. Monitoring:
       - Track query execution times
       - Monitor bytes scanned
       - Set up CloudWatch alerts
    """)